## Projet Deep Learning : Plateforme intelligente d'analyse d'imagerie médicale des plaies
### Partie 2 : Extraction de caractéristiques, Recherche par similarité (CBIR) & Détection OOD


### 1. Exercice 2.1 : Extraction de Features (Embeddings)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

!pip install chromadb

import chromadb


In [ ]:

# Définition du device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Utilisation du device : {device}")

# Chemin de votre dataset sur le Drive monté
DRIVE_DATASET_DIR = '/content/drive/MyDrive/dataset'
IMAGE_SIZE = 224

# Transformations standards pour l'extraction (ResNet50)
val_test_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

### 2. Chargement des données et préparation des chemins d'accès

In [ ]:
# Chargement du fichier CSV contenant les métadonnées des images
data = pd.read_csv('data.csv')

# Création de la colonne 'path' pour le chemin complet de l'image
# et 'filename' qui sera utilisé par index_historical_cases (si besoin)
data['path'] = data.apply(lambda row: os.path.join(DRIVE_DATASET_DIR, row['Class'], row['Name_img']), axis=1)
data['filename'] = data['Name_img']

print(f"Exemple de chemin d'image: {data['path'].iloc[0]}")


In [ ]:
# Filter out non-existent image paths from the DataFrame
initial_data_size = len(data)
data = data[data['path'].apply(os.path.exists)]
filtered_data_size = len(data)

if initial_data_size != filtered_data_size:
    print(f"Warning: {initial_data_size - filtered_data_size} images were removed from the dataset "
          f"because their paths did not exist. Remaining images: {filtered_data_size}.")
else:
    print(f"All {initial_data_size} image paths verified to exist.")

# Afficher les 12 premières images pour vérifier
fig, axes = plt.subplots(3, 4, figsize=(15, 10))
axes = axes.flatten()

sample_data = data.head(12)
for idx, (_, row) in enumerate(sample_data.iterrows()):
    if idx >= len(axes):
        break
    img_path = row['path']
    if os.path.exists(img_path):
        img = Image.open(img_path)
        axes[idx].imshow(img)
        axes[idx].set_title(row['Class'], fontsize=10)
    else:
        axes[idx].text(0.5, 0.5, f"Image non trouvée: {os.path.basename(img_path)}", ha='center', va='center', fontsize=8, color='red')
    axes[idx].axis('off')

for j in range(idx + 1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()



###2. Exercice 2.1 : Extraction des Embeddings Visuels via ResNet50


In [ ]:
class FeatureExtractor(nn.Module):
    def __init__(self):
        super(FeatureExtractor, self).__init__()
        # Tronc commun ResNet50 pré-entraîné
        base_model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        # On coupe la dernière couche fully-connected pour récupérer l'espace latent (2048 dim)
        self.features = nn.Sequential(*list(base_model.children())[:-1])

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        return x

extractor = FeatureExtractor().to(device)
extractor.eval()

# %% [markdown]
# ## 3. Exercice 2.2 : Création de la Base Vectorielle (ChromaDB) et Indexation du Drive

# %%

In [ ]:
# Initialisation de la base de données vectorielle persistante
chroma_client = chromadb.PersistentClient(path="./chroma_db")

# On utilise la similarité cosinus pour comparer les caractéristiques des plaies
collection = chroma_client.get_or_create_collection(
    name="drive_wounds_collection",
    metadata={"hnsw:space": "cosine"}
)

# Fonction pour scanner le Drive, extraire les features et indexer
def index_drive_dataset(base_dir, model, collection):
    print("Début du scan et de l'indexation du dossier Google Drive...")

    # Parcourt les sous-dossiers (chaque sous-dossier = une classe)
    for root, dirs, files in os.walk(base_dir):
        cls_name = os.path.basename(root)
        if root == base_dir:
            continue

        print(f"Indexation de la classe : {cls_name}...")
        for img_name in files:
            if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                img_path = os.path.join(root, img_name)
                img_id = f"drive_{cls_name}_{img_name}"

                try:
                    # Extraction du vecteur d'embedding
                    image = Image.open(img_path).convert("RGB")
                    tensor = val_test_transforms(image).unsqueeze(0).to(device)

                    with torch.no_grad():
                        embedding = model(tensor).cpu().numpy().flatten().tolist()

                    # Ajout à ChromaDB
                    collection.add(
                        embeddings=[embedding],
                        metadatas=[
                            {
                                "file_name": img_name,
                                "class": cls_name,
                                "image_path": img_path,
                            }
                        ],
                        ids=[img_id],
                    )
                except Exception as e:
                    print(f"Erreur sur le fichier {img_name} : {e}")

    print(f"Indexation achevée. {collection.count()} images répertoriées dans la base vectorielle.")

# Exécuter l'indexation sur votre dossier
index_drive_dataset(DRIVE_DATASET_DIR, extractor, collection)

# %% [markdown]
# ## 4. Exercice 2.3 : Moteur de recherche de Similarité avec Affichage en Grille (Grid)


In [ ]:
def search_and_display_grid(query_image_path, model, collection, top_k=4):
    """
    Prend une image de requête, cherche les top_k correspondances dans ChromaDB,
    et affiche les cas historiques trouvés sous forme de grille Matplotlib.
    """
    model.eval()

    # 1. Extraction des caractéristiques de la requête
    image_query = Image.open(query_image_path).convert("RGB")
    tensor = val_test_transforms(image_query).unsqueeze(0).to(device)

    with torch.no_grad():
        query_embedding = model(tensor).cpu().numpy().flatten().tolist()

    # 2. Requête dans ChromaDB
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )

    # 3. Affichage en Grille (Grid) avec Matplotlib
    # On ajoute +1 à la grille pour afficher l'image de requête en première position
    n_plots = top_k + 1
    n_cols = 5 if n_plots >= 5 else n_plots
    n_rows = (n_plots + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
    axes = axes.flatten() if isinstance(axes, np.ndarray) else [axes]

    # Case 0 : Image Requête originale du clinicien
    axes[0].imshow(image_query)
    axes[0].set_title("[REQUÊTE] Image clinique", fontsize=10, color='blue')
    axes[0].axis('off')

    # Cases suivantes : Les cas similaires trouvés dans le Drive historique
    for i in range(top_k):
        ax_idx = i + 1
        if ax_idx >= len(axes):
            break

        metadata = results['metadatas'][0][i]
        distance = results['distances'][0][i]
        similarity = 1 - distance  # Score de corrélation cosinus
        historical_path = metadata['image_path']

        if os.path.exists(historical_path):
            img_hist = Image.open(historical_path)
            axes[ax_idx].imshow(img_hist)
            axes[ax_idx].set_title(
                f"Top {i+1} : {metadata['class']}\nSimil: {similarity:.2%}",
                fontsize=10,
                color='green'
            )
        else:
            axes[ax_idx].text(0.5, 0.5, "Image manquante\nsur le Drive", ha='center', va='center')

        axes[ax_idx].axis('off')

    # Nettoyage des sous-graphiques en excès
    for j in range(n_plots, len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    plt.show()

# Exemple d'utilisation du moteur de recherche visuel :
# search_and_display_grid('/content/test_wound.jpg', extractor, collection, top_k=4)

### Démonstration du Moteur de Recherche Visuel (CBIR)

In [ ]:
# Exemple d'utilisation du moteur de recherche visuel :
# on s'assure que chroma db est bien indexé d'abord :).
# Remplacez le chemin ci-dessous par un chemin d'image réel de votre dataset.
example_query_image_path = os.path.join(DRIVE_DATASET_DIR, 'Abrasions', 'abrasions (1).jpg')

if os.path.exists(example_query_image_path):
    search_and_display_grid(example_query_image_path, extractor, collection, top_k=4)
else:
    print(f"L'image de requête d'exemple n'existe pas: {example_query_image_path}")
    print("Veuillez modifier 'example_query_image_path' avec un chemin valide de votre dataset.")


### Téléchargement des artefacts

In [ ]:
# Sauvegarde du modèle CNN (FeatureExtractor)
os.makedirs("models", exist_ok=True)
torch.save(extractor.state_dict(), "models/cnn_feature_extractor.pth")
print("Modèle CNN (FeatureExtractor) sauvegardé sous models/cnn_feature_extractor.pth")

In [ ]:
import json

# Création du mapping des classes
unique_classes = data['Class'].unique().tolist()
class_mapping = {str(i): cls for i, cls in enumerate(unique_classes)}

# Sauvegarde du mapping des classes
with open("class_mapping.json", "w") as f:
    json.dump(class_mapping, f, indent=4)
print("Mapping des classes sauvegardé sous class_mapping.json")

You can download the following artifacts from your Colab environment:

1.  **Le modèle CNN sauvegardé**: `models/cnn_feature_extractor.pth`
2.  **Le mapping des classes**: `class_mapping.json`
3.  **L'autoencoder sauvegardé**: `models/wound_autoencoder.pth`
4.  **Les embeddings indexés dans ChromaDB**: Le dossier entier `chroma_db`

**Note sur la base de connaissances médicales indexée dans ChromaDB**: D'après le notebook actuel, seule la base d'images des plaies a été indexée dans ChromaDB. Il n'y a pas de base de connaissances textuelle médicale indexée séparément.

Pour télécharger ces fichiers, vous pouvez:
*   Utiliser le panneau de fichiers à gauche dans Colab, naviguer vers le fichier/dossier, faire un clic droit et sélectionner 'Télécharger'.
*   Utiliser le code Python suivant dans une nouvelle cellule pour télécharger programmatiquement:

```python
from google.colab import files

# Pour un fichier spécifique
files.download('models/cnn_feature_extractor.pth')
files.download('class_mapping.json')
files.download('models/wound_autoencoder.pth')

# Pour un dossier entier (nécessite de le zipper d'abord)
!zip -r /content/chroma_db.zip /content/chroma_db
files.download('/content/chroma_db.zip')
```

# %% [markdown]
# ## 5. Module de Sécurité Optionnel : Auto-encodeur pour le filtrage Hors-Domaine (OOD)

In [ ]:
class WoundAutoencoder(nn.Module):
    def __init__(self):
        super(WoundAutoencoder, self).__init__()
        # Compresse l'image d'entrée
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1),  # (32, 112, 112)
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1), # (64, 56, 56)
            nn.BatchNorm2d(64),
            nn.ReLU()
        )
        # Décompresse et reconstruit l'image originale
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1), # (32, 112, 112)
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 3, kernel_size=3, stride=2, padding=1, output_padding=1),  # (3, 224, 224)
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

autoencoder = WoundAutoencoder().to(device)
print("Auto-encodeur OOD prêt à être configuré.")

In [ ]:
# For the autoencoder, transforms should not standardize with ImageNet
# in order to use MSELoss or BCELoss directly on pixels [0, 1]
ae_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor()
])


In [ ]:
# Custom Dataset for Autoencoder Training
class WoundDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.iloc[idx]['path']
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        # For autoencoder, the target is the input itself
        return image, image

# Instantiate the dataset and dataloader for autoencoder training
train_dataset_ae = WoundDataset(data, transform=ae_transforms)
train_loader_ae = DataLoader(train_dataset_ae, batch_size=32, shuffle=True)

print(f"WoundDataset for Autoencoder created with {len(train_dataset_ae)} images.")


### Entraînement de l'Auto-encodeur (OOD)

In [ ]:
def train_autoencoder(model, dataloader, epochs=10, lr=0.001):
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    print("Début de l'entraînement de l'Auto-encodeur (OOD)...")
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0

        for inputs, _ in dataloader:
            inputs = inputs.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, inputs) # La cible est l'image d'origine elle-même
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * inputs.size(0)

        epoch_loss = total_loss / len(dataloader.dataset)
        print(f"Époque [{epoch+1}/{epochs}] - Perte de reconstruction MSE: {epoch_loss:.4f}")

    os.makedirs("models", exist_ok=True)
    torch.save(model.state_dict(), "models/wound_autoencoder.pth")
    print("Modèle Auto-encodeur sauvegardé.")

# Train the autoencoder
train_autoencoder(autoencoder, train_loader_ae, epochs=10, lr=0.001)

### Définition du Seuil de Rejet OOD (Calculé sur la validation ID)

In [ ]:
def compute_ood_threshold(model, val_loader_ae, percentile=95):
    """
    Calcule l'erreur de reconstruction sur le jeu de validation de plaies (In-Distribution).
    Le seuil est fixé au Xème percentile des erreurs les plus élevées.
    """
    model.eval()
    criterion = nn.MSELoss(reduction='none') # Erreur par image
    errors = []

    with torch.no_grad():
        for inputs, _ in val_loader_ae:
            inputs = inputs.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, inputs)
            # Moyenne de l'erreur par échantillon (sur les canaux, largeurs et hauteurs)
            sample_losses = loss.mean(dim=[1, 2, 3]).cpu().numpy()
            errors.extend(sample_losses)

    threshold = np.percentile(errors, percentile)
    print(f"Seuil de détection OOD défini (Percentile {percentile}) : {threshold:.5f}")
    return threshold


### Pipeline de Validation OOD (Test clinique final)

In [ ]:
def verify_image_domain(image_path, model, threshold):
    """
    Prend une image en entrée, calcule son erreur de reconstruction.
    Si l'erreur > threshold -> Rejet de l'image (Out-Of-Distribution).
    """
    model.eval()

    image = Image.open(image_path).convert("RGB")
    tensor = ae_transforms(image).unsqueeze(0).to(device)

    criterion = nn.MSELoss() # Changed from MSELoss(reduction='none') to MSELoss()
    with torch.no_grad():
        reconstruction = model(tensor)
        reconstruction_error = criterion(reconstruction, tensor).item()

    is_in_distribution = reconstruction_error <= threshold

    print(f"\n--- Vérification de domaine pour : {os.path.basename(image_path)} ---")
    print(f"Erreur de reconstruction : {reconstruction_error:.5f} (Seuil maximal autorisé : {threshold:.5f})")
    if is_in_distribution:
        print("Verdict : [VALIDE] Image médicale de plaie détectée. Traitement autorisé.")
    else:
        print("Verdict : [ALERTE OOD] Contenu non conforme. Traitement bloqué par l'assistant.")

    return is_in_distribution
